<h1 style="color:DodgerBlue">Индивидуальный проект</h1>

## Название проекта: отзывы о продуктах и услугах

**Вариант №18. Студент: Чирченко Максим.**

Продолжение проекта Sprint1: базовый `Review`, производные `ProductReview`, `ServiceReview`, `ServiceTimeReview`.
Сохраняются операции `DisplayReview`, `RateProduct`, `GetReviewDetails`.

### Дополнительное задание
Добавить к существующим базовому и производным классам по 3–4 атрибута и метода; реализовать простое, сложное и множественное наследование.

### Описание решения

| Класс | Добавленные атрибуты | Добавленные методы |
|---|---|---|
| Review | Author, CreatedAt, IsPublished, HelpfulVotes | EditText, Publish, Hide, MarkHelpful |
| ProductReview | ProductName, PurchaseDate, IsVerifiedPurchase | RenameProduct, SetPurchaseDate, ConfirmPurchase |
| ServiceReview | ServiceName, EmployeeName, ServiceCost | Reschedule, AssignEmployee, UpdateServiceCost |
| ServiceTimeReview | EndTime, WaitingMinutes, WasOnTime | RescheduleTime, CalculateDuration, RecordWaiting |

Простое наследование: `Review → ProductReview` и `Review → ServiceReview`.
Сложное наследование: `Review → ServiceReview → ServiceTimeReview`.
Множественное наследование контрактов: `Review : IRateable, IPublishable`; производные классы наследуют обе реализации.
`Review` — абстрактный класс с общей реализацией, а `DisplayReview` реализуется наследниками.

Некорректные оценки, отрицательная стоимость и неверный временной интервал отклоняются исключениями.
Продолжительность услуги вычисляется в пределах одного дня.

<h2 style="color:DodgerBlue">Реализация</h2>

In [1]:
using System;
using System.Collections.Generic;

public interface IRateable
{
    int Rating { get; }
    void RateProduct(int rating);
}
public interface IPublishable
{
    bool IsPublished { get; }
    void Publish();
    void Hide();
}

// Общая основа отзывов и реализация двух независимых контрактов.
public abstract class Review : IRateable, IPublishable
{
    public int ReviewId { get; }
    public string Text { get; private set; }
    public int Rating { get; private set; }
    public string Author { get; set; }
    public DateTime CreatedAt { get; }
    public bool IsPublished { get; private set; }
    public int HelpfulVotes { get; private set; }

    protected Review(int reviewId, string text, int rating, string author)
    {
        ReviewId = reviewId;
        EditText(text);
        RateProduct(rating);
        Author = author;
        CreatedAt = DateTime.Today;
    }
    public virtual void RateProduct(int rating)
    {
        if (rating < 1 || rating > 5)
            throw new ArgumentOutOfRangeException(nameof(rating), "Оценка должна быть от 1 до 5.");
        Rating = rating;
    }
    public void EditText(string text)
    {
        if (string.IsNullOrWhiteSpace(text))
            throw new ArgumentException("Текст отзыва не должен быть пустым.", nameof(text));
        Text = text;
    }
    public void Publish() => IsPublished = true;
    public void Hide() => IsPublished = false;
    public void MarkHelpful() => HelpfulVotes++;
    public abstract void DisplayReview();
    public virtual string GetReviewDetails() =>
        $"Отзыв #{ReviewId}: {Text}; {Rating}/5; автор: {Author}; создан: {CreatedAt:dd.MM.yyyy}; опубликован: {IsPublished}; полезен: {HelpfulVotes}";
}

// Простое наследование: Review -> ProductReview.
public class ProductReview : Review
{
    public string ProductId { get; }
    public string ProductName { get; private set; }
    public DateTime PurchaseDate { get; private set; }
    public bool IsVerifiedPurchase { get; private set; }

    public ProductReview(int id, string text, int rating, string productId,
        string author, string productName, DateTime purchaseDate)
        : base(id, text, rating, author)
    {
        ProductId = productId;
        RenameProduct(productName);
        SetPurchaseDate(purchaseDate);
    }
    public void RenameProduct(string name)
    {
        if (string.IsNullOrWhiteSpace(name)) throw new ArgumentException("Название товара обязательно.", nameof(name));
        ProductName = name;
    }
    public void SetPurchaseDate(DateTime date)
    {
        if (date.Date > DateTime.Today) throw new ArgumentOutOfRangeException(nameof(date), "Дата покупки не может быть в будущем.");
        PurchaseDate = date.Date;
        IsVerifiedPurchase = false;
    }
    public void ConfirmPurchase() => IsVerifiedPurchase = true;
    public override void DisplayReview() => Console.WriteLine($"Товар «{ProductName}»: {Text} ({Rating}/5)");
    public override string GetReviewDetails() =>
        $"{base.GetReviewDetails()}; товар: {ProductId}, {ProductName}; покупка: {PurchaseDate:dd.MM.yyyy}; подтверждена: {IsVerifiedPurchase}";
}

// Простое наследование: Review -> ServiceReview.
public class ServiceReview : Review
{
    public DateTime VisitDate { get; private set; }
    public string ServiceName { get; set; }
    public string EmployeeName { get; private set; }
    public decimal ServiceCost { get; private set; }

    public ServiceReview(int id, string text, int rating, DateTime visitDate,
        string author, string serviceName, string employeeName, decimal serviceCost)
        : base(id, text, rating, author)
    {
        VisitDate = visitDate.Date;
        ServiceName = serviceName;
        AssignEmployee(employeeName);
        UpdateServiceCost(serviceCost);
    }
    public void Reschedule(DateTime date) => VisitDate = date.Date;
    public void AssignEmployee(string name)
    {
        if (string.IsNullOrWhiteSpace(name)) throw new ArgumentException("Имя сотрудника обязательно.", nameof(name));
        EmployeeName = name;
    }
    public void UpdateServiceCost(decimal cost)
    {
        if (cost < 0) throw new ArgumentOutOfRangeException(nameof(cost), "Стоимость не может быть отрицательной.");
        ServiceCost = cost;
    }
    public override void DisplayReview() => Console.WriteLine($"Услуга «{ServiceName}» от {VisitDate:dd.MM.yyyy}: {Text} ({Rating}/5)");
    public override string GetReviewDetails() =>
        $"{base.GetReviewDetails()}; услуга: {ServiceName}; посещение: {VisitDate:dd.MM.yyyy}; сотрудник: {EmployeeName}; стоимость: {ServiceCost:F2} руб.";
}

// Многоуровневое наследование: Review -> ServiceReview -> ServiceTimeReview.
public class ServiceTimeReview : ServiceReview
{
    public TimeSpan StartTime { get; private set; }
    public TimeSpan EndTime { get; private set; }
    public int WaitingMinutes { get; private set; }
    public bool WasOnTime { get; private set; }

    public ServiceTimeReview(int id, string text, int rating, DateTime date, TimeSpan startTime,
        string author, string serviceName, string employeeName, decimal cost, TimeSpan endTime)
        : base(id, text, rating, date, author, serviceName, employeeName, cost)
    {
        RescheduleTime(startTime, endTime);
    }
    // Время начала и окончания относится к одному дню.
    public void RescheduleTime(TimeSpan start, TimeSpan end)
    {
        if (start < TimeSpan.Zero || end >= TimeSpan.FromDays(1) || end <= start)
            throw new ArgumentException("Нужно указать время одного дня, окончание позже начала.");
        StartTime = start;
        EndTime = end;
        WaitingMinutes = 0;
        WasOnTime = true;
    }
    public TimeSpan CalculateDuration() => EndTime - StartTime;
    public void RecordWaiting(int minutes)
    {
        if (minutes < 0) throw new ArgumentOutOfRangeException(nameof(minutes));
        WaitingMinutes = minutes;
        WasOnTime = minutes == 0;
    }
    public override void DisplayReview()
    {
        base.DisplayReview();
        Console.WriteLine($"Время: {StartTime:hh\\:mm}–{EndTime:hh\\:mm}; ожидание: {WaitingMinutes} мин.");
    }
    public override string GetReviewDetails() =>
        $"{base.GetReviewDetails()}; время: {StartTime:hh\\:mm}–{EndTime:hh\\:mm}; длительность: {CalculateDuration().TotalMinutes} мин.; ожидание: {WaitingMinutes} мин.; вовремя: {WasOnTime}";
}

The below script needs to be able to find the current output cell; this is an easy method to get it.

### Демонстрация всех видов наследования и новых методов

In [2]:
// Создание и взаимодействие объектов варианта №18.
var product = new ProductReview(1, "Удобная клавиатура", 4, "P-018", "Максим",
    "Клавиатура", new DateTime(2026, 9, 18));
product.RenameProduct("Механическая клавиатура");
product.SetPurchaseDate(new DateTime(2026, 9, 19));
product.ConfirmPurchase();
product.EditText("Удобная механическая клавиатура, тихие клавиши");
product.MarkHelpful();

var service = new ServiceReview(2, "Хорошая стрижка", 4, new DateTime(2026, 9, 18),
    "Анна", "Стрижка", "Ольга", 900m);
service.Reschedule(new DateTime(2026, 9, 19));
service.AssignEmployee("Елена");
service.UpdateServiceCost(1000m);

var timed = new ServiceTimeReview(3, "Быстро отремонтировали телефон", 4,
    new DateTime(2026, 9, 20), new TimeSpan(14, 0, 0), "Олег",
    "Ремонт телефона", "Иван", 1500m, new TimeSpan(15, 0, 0));
timed.RescheduleTime(new TimeSpan(14, 30, 0), new TimeSpan(15, 15, 0));
timed.RecordWaiting(5);
Console.WriteLine($"Продолжительность ремонта: {timed.CalculateDuration().TotalMinutes} мин.");

var reviews = new List<Review> { product, service, timed };
foreach (Review review in reviews)
{
    // Один объект используется через два интерфейса.
    IRateable rateable = review;
    rateable.RateProduct(5);
    IPublishable publishable = review;
    publishable.Publish();
    review.DisplayReview();
    Console.WriteLine(review.GetReviewDetails());
    Console.WriteLine();
}
IPublishable publication = product;
publication.Hide();
Console.WriteLine($"Отзыв о товаре скрыт: {!publication.IsPublished}");
publication.Publish();

Продолжительность ремонта: 45 мин.


Товар «Механическая клавиатура»: Удобная механическая клавиатура, тихие клавиши (5/5)


Отзыв #1: Удобная механическая клавиатура, тихие клавиши; 5/5; автор: Максим; создан: 18.10.2026; опубликован: True; полезен: 1; товар: P-018, Механическая клавиатура; покупка: 19.09.2026; подтверждена: True


Услуга «Стрижка» от 19.09.2026: Хорошая стрижка (5/5)


Отзыв #2: Хорошая стрижка; 5/5; автор: Анна; создан: 18.10.2026; опубликован: True; полезен: 0; услуга: Стрижка; посещение: 19.09.2026; сотрудник: Елена; стоимость: 1000,00 руб.


Услуга «Ремонт телефона» от 20.09.2026: Быстро отремонтировали телефон (5/5)


Время: 14:30–15:15; ожидание: 5 мин.


Отзыв #3: Быстро отремонтировали телефон; 5/5; автор: Олег; создан: 18.10.2026; опубликован: True; полезен: 0; услуга: Ремонт телефона; посещение: 20.09.2026; сотрудник: Иван; стоимость: 1500,00 руб.; время: 14:30–15:15; длительность: 45 мин.; ожидание: 5 мин.; вовремя: False


Отзыв о товаре скрыт: True
